In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import anndata as ad
#import leidenalg
import scrublet as scr
import seaborn as sns
import matplotlib
import tqdm.auto as tqdm
#from statannotations.Annotator import Annotator
#import biomart


import re

In [ ]:
%matplotlib  inline

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

## Load Data

In [ ]:
data_paths = ['/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg18_soupx.h5ad',
                    '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg20_soupx.h5ad',
                    '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg24_soupx.h5ad',
                    '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg26_soupx.h5ad',
                    '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sg28_soupx.h5ad'
                   ]
adatas_in_memory = []

for file_path in data_paths:
    adata_backed = sc.read_h5ad(file_path, backed='r')

    # Load full data into memory for modifications
    adata_mem = adata_backed.to_memory()
    
    # Rename obs column
    if 'orig.ident' in adata_mem.obs.columns:
        adata_mem.obs.rename(columns={'orig.ident': 'sample'}, inplace=True)
    
    # Make unique names
    adata_mem.var_names_make_unique()
    adata_mem.obs_names_make_unique()
    
    adatas_in_memory.append(adata_mem)

# Concatenate all in-memory AnnData objects
adata = ad.concat(adatas_in_memory, join='outer', label='batch', keys=[f'file{i}' for i in range(len(adatas_in_memory))])

# Now you can proceed further
adata.obs = adata.obs.rename(columns={'orig.ident': 'sample'})

# Ensure unique cell and gene names
adata.var_names_make_unique()
adata.obs_names_make_unique()

In [ ]:
samples = {"/data/projects/manuela/scRNA_aging-mouse/sg18_soupx.h5ad": "SG18",
      "/data/projects/manuela/scRNA_aging-mouse/sg20_soupx.h5ad": "SG20",
       "/data/projects/manuela/scRNA_aging-mouse/sg24_soupx.h5ad": "SG24",
       "/data/projects/manuela/scRNA_aging-mouse/sg26_soupx.h5ad": "SG26",
       "/data/projects/manuela/scRNA_aging-mouse/sg28_soupx.h5ad": "SG28"
    }

In [ ]:
adata.obs['sample'] = adata.obs['sample'].replace(samples)
adata.obs

# 1. Adata Shape Beginning

## Save nb. cells per sample -> step beginning

In [ ]:
#Cells per sample
cpsample_beg=adata.obs['sample'].value_counts()
cpsample_beg

# Cells per sample per cluster
#cpspc_beg = adata.obs['clusters'].value_counts()

## Save Median of nb. Genes per Cell per Sample -> step beginning

In [ ]:
#Median of Genes per Cell per Sample
gpsample_beg=adata.obs.groupby('sample')['nFeature_RNA'].median()
gpsample_beg

# Scrublet

In [ ]:
results = []

for sample in tqdm.tqdm(adata.obs["sample"].unique()):
    tmp = adata[adata.obs["sample"] == sample]
    counts_matrix = tmp.X
    #Initialize Scrublet object
    scrub = scr.Scrublet(counts_matrix)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(use_approx_neighbors=False)
    res = pd.DataFrame({"doublet_scores": doublet_scores, "predicted_doublets": predicted_doublets,},index=tmp.obs_names,)
    scrub.plot_histogram()
    
    scrub.set_embedding("UMAP", scr.get_umap(scrub.manifold_obs_))
    scrub.plot_embedding("UMAP", order_points=True)
    
    results.append(res)
    
results = pd.concat(results, axis=0)
results = results.reindex(adata.obs_names)
adata.obs['doublets'] = results['predicted_doublets']

# Filter out predicted doublets
adata = adata[~adata.obs['doublets'].astype(bool), :]

results.to_csv("/data/projects/manuela/aging/scRNA_aging-mouse/scrublet-aging_soupxed.csv", index=True)
print(results.predicted_doublets.value_counts())

In [ ]:
# Check if there is no true doublet remaining
adata[adata.obs['doublets']==True]

# 2. Adata.shape after Scrublet

In [ ]:
#Shape after Scrublet
adata.shape

## Save nb. cells per sample -> step 2

In [ ]:
#Cells per sample
cpsample_2=adata.obs['sample'].value_counts()
cpsample_2

# Normalize & Integrate

In [ ]:
adata.var_names_make_unique()  # this is unnecessary if using `var_names='gene_ids'` in `sc.read_10x_mtx`
adata.obs_names_make_unique()

In [ ]:
#adata.obs

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, 
                         # save='-hp1ab-sg7_highest_expr_genes.png'
                        )

In [ ]:
#sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_cells(adata, min_genes=400)
sc.pp.filter_genes(adata, min_cells=3) 

In [ ]:
#adata.obs

## 3. Adata Shape after filtering min_genes=200, min_cells=3

In [ ]:
#Shape after filtering min_genes=200, min_cells=3
adata.shape

### Save nb. cells per sample -> step 3

In [ ]:
#Cells per sample
#-> no reduction of cells
cpsample_3=adata.obs['sample'].value_counts()
cpsample_3

### Save median genes per sample -> step 3

In [ ]:
#adata.var['mt'] = adata.var_names.isin(mt_list)
adata.var['mt'] = adata.var_names.str.startswith('mt-')  # annotate the group of mitochondrial genes as 'mt'
adata.var[adata.var['mt']==True]

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
adata.obs

In [ ]:
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

In [ ]:
def plot_violin(adata):
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], jitter=0.4, multi_panel=True, groupby = 'sample', rotation = 90
                 # save='-hp1ab-sg7.png'
    )
    sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt',
                  # save='hp1ab-sg7_scatter_pct_mt.png'
                 )
    sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts',
                  # save='-hp1ab-sg7_scatter_genes_counts.png'
                 )

In [ ]:
plot_violin(adata)

Fig Correlation Filter n_genes_by_counts < 2500 for each Method

In [ ]:
#adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.n_genes_by_counts < 5000, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

## 4. Adata Shape after filtering n_genes_by_counts < 5000 & pct_counts_mt < 5

### Save nb. cells per sample -> step 4

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
#sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, n_top_genes=2000)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

In [ ]:
# Create a pattern to match any variation of qpct/qpctl
genes_of_interest = ['Qpct', 'Qpctl', 'Ccl2']  # Add Ccl2 to your gene list
matching_genes = [gene for gene in adata.var_names if gene in genes_of_interest]

print("Matching genes:", matching_genes)
gene_of_interest = matching_genes

sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale='var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale='var')

# Check if both genes are in adata.raw.var_names (original full set)
print("Genes in adata.raw:", [gene for gene in adata.var_names if gene.lower().startswith('qpct')])

# Are both genes (Qpct, Qpctl) actually highly variable?
print("Is Qpctl highly variable?:", adata.var.loc['Qpctl', 'highly_variable'])
print("Is Qpct highly variable?:", adata.var.loc['Qpct', 'highly_variable'])
print("Is Ccl2 highly variable?:", adata.var.loc['Ccl2', 'highly_variable'])

In [ ]:
adata.raw = adata

# Ensure 'Qpctl' stays in var_names
adata.var.loc['Qpctl', 'highly_variable'] = True

In [ ]:
# Do the filtering
adata = adata[:, adata.var.highly_variable]

In [ ]:
# Create a pattern to match any variation of qpct/qpctl
pattern = re.compile(r'Qpct', re.IGNORECASE)

# Search for matches in adata.raw.var_names
matching_genes = [gene for gene in adata.var_names if pattern.search(gene)]

# Print the matching genes
print("Matching genes:", matching_genes)
gene_of_interest = matching_genes

# Create the violin plot
sc.pl.violin(adata, keys=gene_of_interest, groupby='sample', rotation=90)
sc.pl.dotplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')
sc.pl.matrixplot(adata, var_names=gene_of_interest, groupby='sample', standard_scale = 'var')

# Check if both genes are in adata.raw.var_names (original full set)
print("Genes in adata.raw:", [gene for gene in adata.var_names if gene.lower().startswith('qpct')])

# Are both genes (Qpct, Qpctl) actually highly variable?
print("Is Qpctl highly variable?:", adata.var.loc['Qpctl', 'highly_variable'])
print("Is Qpct highly variable?:", adata.var.loc['Qpct', 'highly_variable'])

In [ ]:
#After cutting to top 2000 highly variable genes 
adata.shape

In [ ]:
# Regress out effects of total counts per cell and the percentage of mitochondrial genes expressed. Scale the data to unit variance.
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

In [ ]:
# Scale each gene to unit variance. Clip values exceeding standard deviation 10.
sc.pp.scale(adata, max_value=10)

# PCA

In [ ]:
# Reduce the dimensionality of the data by running principal component analysis (PCA)
sc.tl.pca(adata, svd_solver='arpack',use_highly_variable=True, n_comps=50) #flag use high var gene -> no throw out of data'

In [ ]:
def plot_pca(adata):
    sc.pl.pca(adata, color = 'sample' 
              # save='-hp1ab-sg7_pca_2dmap.png'
             )
    sc.pl.pca_variance_ratio(adata, log=True, 
                             # save='-hp1ab-sg7_PC_components.png'
                            )
plot_pca(adata)

In [ ]:
sc.external.pp.harmony_integrate(adata=adata, key='sample', basis='X_pca')

# UMAP before integration

In [ ]:
sc.pp.neighbors(adata, n_pcs=20, use_rep="X_pca")
sc.tl.umap(adata)
sc.pl.umap(adata, color=['sample', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt'], show = False)

## Computing the neighborhood graph

In [ ]:
#sc.pp.neighbors(adata, n_pcs=20, use_rep="X_pca_harmony")
sc.pp.neighbors(adata, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")

## Clustering w. Leiden

In [ ]:
sc.tl.leiden(adata, key_added='clusters', resolution=0.31)

# Plot Integrated

In [ ]:
sc.tl.umap(adata)
sc.pl.umap(adata, color=['sample', 'clusters', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt'])

## Embedding the neighborhood graph

In [ ]:
# compute clusters using the leiden method and store the results with the name `clusters`
#sc.tl.leiden(adata, key_added='clusters', resolution=0.31)
#sc.tl.umap(adata)

fig = sc.pl.umap(adata, color=['clusters'], show = False)
#fig.legend()

In [ ]:
adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['sample', 'clusters'], wspace=0.4)

In [ ]:
outfile = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/integrated-aging-soupxed_withQpctl.h5ad'
adata.write(outfile)

# Subclustering

In [ ]:
#sub2 = adata[adata.obs.clusters=='2']
#sub3 = adata[adata.obs.clusters=='3']
sub4 = adata[adata.obs.clusters=='4']
sub12 = adata[adata.obs.clusters=='12']
sub14 = adata[adata.obs.clusters=='14']
#sub13 = adata[adata.obs.clusters=='13']
sub12

Neighbors

In [ ]:
#sc.pp.neighbors(sub2, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
#sc.pp.neighbors(sub3, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
sc.pp.neighbors(sub4, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
sc.pp.neighbors(sub12, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
#sc.pp.neighbors(sub13, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")
sc.pp.neighbors(sub14, n_pcs=20, metric="cosine", n_neighbors=30, use_rep="X_pca_harmony")

Clustering

In [ ]:
#sc.tl.leiden(sub2, key_added='subclusters', resolution=0.31)
#sc.tl.leiden(sub3, key_added='subclusters', resolution=0.31)
sc.tl.leiden(sub4, key_added='subclusters', resolution=0.31)
sc.tl.leiden(sub12, key_added='subclusters', resolution=0.31)
#sc.tl.leiden(sub13, key_added='subclusters', resolution=0.31)
sc.tl.leiden(sub14, key_added='subclusters', resolution=0.31)

UMAPs

In [ ]:
#fig = sc.pl.umap(sub2, color=['subclusters'], title= "subclutser of cluster 2",show = False)
#fig = sc.pl.umap(sub3, color=['subclusters'], title= "subclutser of cluster 3 ",show = False)
fig = sc.pl.umap(sub4, color=['subclusters'],  title= "subclutser of cluster 4",show = False)
fig = sc.pl.umap(sub12, color=['subclusters'], title= "subclutser of cluster 12",show = False)
#fig = sc.pl.umap(sub13, color=['subclusters'], title= "subclutser of cluster 13",show = False)
fig = sc.pl.umap(sub14, color=['subclusters'], title= "subclutser of cluster 14",show = False)

In [ ]:
set(sub12.obs['subclusters'])

Save objects

In [ ]:
#outsub2 = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub2_integrated-aging-soupxed.h5ad'
#outsub3 = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub3_integrated-aging-soupxed.h5ad'
outsub4 = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub4_integrated-aging-soupxedQpctl.h5ad'
outsub12 = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub12_integrated-aging-soupxedQpctl.h5ad'
#outsub13 = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub13_integrated-aging-soupxed.h5ad'
outsub14 = '/data/projects/manuela/aging/scRNA_aging-mouse/anndata_objects/sub14_integrated-aging-soupxedQpctl.h5ad'

In [ ]:
#sub2.write(outsub2)
#sub3.write(outsub3)
sub4.write(outsub4)
sub12.write(outsub12)
#sub13.write(outsub13)
sub14.write(outsub14)

## Nb Cells per samlpe end -> step end

In [ ]:
#Cells per sample
cpsample_end= adata.obs['sample'].value_counts()
#print(cpsample_beg)
print(cpsample_end)

# Cells per sample per cluster
cpspc_end = adata.obs['clusters'].value_counts()
#print(cpspc_beg)
print(cpspc_end)

In [ ]:
# Set the width of the bars
barWidth = 0.35

# Set the x position of the bars
r1 = np.arange(len(cpsample_beg))
r2 = [x + barWidth for x in r1]

# Create the barplot
plt.bar(r1, cpsample_beg, width=barWidth, edgecolor='white', label='r1')
plt.bar(r2, cpsample_end, width=barWidth, edgecolor='white', label='r2')

# Add labels and titles
plt.xticks([r + barWidth/2 for r in range(len(r1))])
plt.xlabel('X Label')
plt.ylabel('Counts')
plt.title('Cells per Sample per Cluster before and after filtering')

# Add legend
plt.legend()

# Show the plot
plt.show()

In [ ]:
df_gpsb = pd.DataFrame(gpsample_beg)
df_gpsb = df_gpsb.rename(columns={"nFeature_RNA": "Before filtering"})

#AFTER filtering min_genes=200, min_cells=3
df_gps3 = pd.DataFrame(gpsample_3)
df_gps3= df_gps3.rename(columns={"nFeature_RNA": "After min_genes min_cells"})

df_gpse = pd.DataFrame(gpsample_end)
df_gpse = df_gpse.rename(columns={"nFeature_RNA": "After all filtering"})

df_gps = pd.concat([df_gpsb, df_gps3, df_gpse], axis=1)
df_gps

In [ ]:
gpsb_list = df_gps['Before filtering']
gps3_list = df_gps['After min_genes min_cells']
gpse_list = df_gps['After all filtering']

# Convert to floats
gpsb_list = [float(x) for x in gpsb_list]
gps3_list = [float(x) for x in gps3_list]
gpse_list = [float(x) for x in gpse_list]

print(gpsb_list, sum(gpsb_list))
print(gps3_list, sum(gps3_list))
print(gpse_list, sum(gpse_list))

In [ ]:
bar_width = 0.35

f, ax = plt.subplots(figsize=(15,5))
plt.ylabel('Median Nb. Genes per Cell')
plt.xlabel('Sample')

plt.xticks(range(len(cps_bf_list)),('C2', 'C3', 'SG25', 'N6000', 'PigCC', 'L2A'), rotation=30)

#X_axis = np.arange(len(X))

bar1 = plt.bar(np.arange(len(gpse_list)), gpse_list, bar_width, align='center', color = 'orange', label='After cutting to top 2000 highly variable genes')
bar2 = plt.bar(np.arange(len(gps3_list)) - 0.3, gps3_list, bar_width, align='center', color = 'green' , label='After filtering min_genes min_cells')
bar3 = plt.bar(np.arange(len(gpsb_list)) - 0.6, gpsb_list, bar_width, align='center', color = 'blue' , label='Before Filtering') 

# Add counts above the two bar graphs
for rect in bar1 + bar2 + bar3:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width() / 2.0, height-100, f'{height:.0f}', ha='center', va='bottom')

plt.title('Median Nb. Genes per Cell per Sample')    
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
#plt.tight_layout()
plt.show()